In [ ]:
Proximal Policy Optimization(PPO) Implementation

https://jonathan-hui.medium.com/rl-proximal-policy-optimization-ppo-explained-77f014ec3f12
https://jonathan-hui.medium.com/rl-the-math-behind-trpo-ppo-d12f6c745f33
https://medium.com/analytics-vidhya/coding-ppo-from-scratch-with-pytorch-part-1-4-613dfc1b14c8
https://www.datacamp.com/tutorial/proximal-policy-optimization

In [ ]:
PPO was proposed as a simpler and more efficient alternative to TRPO. PPO clips the ratio of the policies to approximate the trust region without resorting to complex calculations involving KL divergence.

In [1]:
import random
import math
import torch
from torch import nn 
from torch import optim
from collections import deque
from torch.distributions import Categorical

from frozen_lake_environment import (generate_grid_randomly,
                                     FrozenLakeEnvironment,
                                     State)
import numpy as np
from matplotlib import pyplot
from visual_utils import (render_policy_and_value, 
                          animate_policy_value_video,
                          plot_trajectory_history)
import torch.nn.functional as F
from tqdm import tqdm

In [2]:
class Actor(nn.Module):
    """
    Policy network π(a | s; θ).
    Outputs a categorical distribution over discrete actions.
    """
    def __init__(self, state_dim: int, act_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden), 
            nn.ReLU(),
            nn.Linear(hidden, act_dim)
        )

    def forward(self, x: torch.Tensor) -> Categorical:
        logits = self.net(x)
        return Categorical(logits=logits)

    def log_prob(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        return self.forward(x).log_prob(a)
            
    def policy_table(self, env):
        states = [State(s_idx, env.n_cols) for s_idx in range(env.n_states)]
        policy = np.zeros(env.n_states, dtype=np.int8)
        for state in states:
            state_vec = state.get_state_feature_vec(env.n_states)
            state_vec = torch.Tensor(state_vec).unsqueeze(0) # 1 X feat_vec
            with torch.no_grad():
                action_probs = self.forward(state_vec).probs
                policy[state.idx] = torch.argmax(action_probs, dim=-1).item()
        return policy

In [3]:
class Critic(nn.Module):
    """
    state-value network V(s; φ).
    Outputs one scalar per discrete action → index by chosen action.
    """
    def __init__(self, state_dim: int, act_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returns V-value for the input state, shape (batch, 1)."""
        return self.net(x)
    
    def v_table(self, env):
        states = [State(s_idx, env.n_cols) for s_idx in range(env.n_states)]
        V = np.zeros(env.n_states, dtype=np.float32)
        for state in states:
            state_feat_vec = state.get_state_feature_vec(env.n_states)
            state_feat_vec = torch.Tensor(state_feat_vec).unsqueeze(0) # 1 X feat_vec
            with torch.no_grad():
                v_value = self.forward(state_feat_vec).squeeze(1)
            V[state.idx] = v_value.item()
        return V

In [4]:
class PPOAgent:
    def __init__(self, env, actor, critic, lr_actor, lr_critic, gamma,
                 epsilon=0.2, entropy_coeff=0.2, batch_size=32):
        self.env        = env
        self.actor      = actor
        self.critic     = critic
        self.opt_actor  = optim.Adam(self.actor.parameters(),  lr=lr_actor, eps=1e-5)  # FIX 1: eps=1e-5 prevents Adam denominator collapse
        self.opt_critic = optim.Adam(self.critic.parameters(), lr=lr_critic, eps=1e-5)
        self.gamma      = gamma
        self.epsilon    = epsilon
        self.entropy_coeff  = entropy_coeff
        self.batch_size = batch_size

    def calculate_returns(self, rewards, discount_factor):
        returns = []
        cumulative_reward = 0
        for r in reversed(rewards):
            cumulative_reward = r + discount_factor * cumulative_reward 
            returns.append(0, cumuative_reward)
        returns = torch.tensor(returns)
        # normalize returns
        return (returns - returns.mean())/(returns.std() + 1e-8)

    def calculate_advantages(self, returns, values):
        advantages = returns - values
        # Normalize the advantage
        advantages = (advantages - advantages.mean()) / advantages.std()
        return advantages

    def calculate_surrogate_loss(self, log_action_probs_old, log_action_probs_new, epsilon, advantages):
        advantages = advantages.detach()
        policy_ratio = (log_action_probs_new - log_action_probs_old).exp()
        loss_1 = policy_ratio * advantages
        loss_2 = torch.clamp(
                    policy_ratio, min=1.0-epsilon, max=1.0+epsilon
                    )*advantages
        return torch.min(loss_1, loss_2)

    def calculate_losses(self, surrogate_loss, entropy, entropy_coefficient, returns, value_pred):
        entropy_bonus = entropy_coefficient * entropy
        policy_loss = -(surrogate_loss + entropy_bonus).sum()
        value_loss = f.smooth_l1_loss(returns, value_pred).sum()
        return policy_loss, value_loss

    def init_training(self):
        states = []
        actions = []
        actions_log_probability = []
        values = []
        rewards = []
        done = False
        episode_reward = 0
        return states, actions, actions_log_probability, values, rewards, done, episode_reward

    def forward_pass(self):
        states, actions, actions_log_probability, values, rewards, done, episode_reward = self.init_training()
        state = env.find('S')
        s_feat = s.get_state_feature_vec(env.n_states)
        s_feat = torch.FloatTensor(s_feat).unsqueeze(0)
        states.append(s_feat)  
        
        while not done:
            action_pred= self.actor(s_feat)
            value_pred = self.critic(s_feat)
    
            action_prob = f.softmax(action_pred, dim=-1)
            dist = distributions.Categorical(action_prob)
            action = dist.sample()
            
            log_prob_action = dist.log_prob(action)

            result = env.step(state, action)
            reward = result["reward"]          
            next_s = result["new_state"]
            next_s_feat = next_s.get_state_feature_vec(env.n_states)
            done = result["is_terminated"]
                        
            actions.append(action)
            actions_log_probability.append(log_prob_action)
            values.append(value_pred)
            rewards.append(reward)
            episode_reward += reward
            
            s = next_s
            s_feat = next_s_feat
        
        states = torch.cat(states)
        actions = torch.cat(actions)
        actions_log_probability = torch.cat(actions_log_probability)
        values = torch.cat(values).squeeze(-1)
        returns = calculate_returns(rewards, discount_factor)
        advantages = calculate_advantages(returns, values)
        return episode_reward, states, actions, actions_log_probability, advantages, returns

In [5]:
def update_policy(agent, states, actions, actions_log_probability_old,
                  advantages, returns, optimizer, ppo_steps, epsilon, entropy_coefficient):
    BATCH_SIZE = 128
    total_policy_loss = 0
    total_value_loss = 0
    actions_log_probability_old = actions_log_probability_old.detach()
    actions = actions.detach()
    training_results_dataset = TensorDataset(
            states,
            actions,
            actions_log_probability_old,
            advantages,
            returns)
    batch_dataset = DataLoader(
            training_results_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False)
    for _ in range(ppo_steps):
        for batch_idx, (states, actions, actions_log_probability_old, advantages, returns) in enumerate(batch_dataset):
            # get new log prob of actions for all input states
            action_pred, value_pred = agent(states)
            value_pred = value_pred.squeeze(-1)
            action_prob = f.softmax(action_pred, dim=-1)
            probability_distribution_new = distributions.Categorical(
                    action_prob)
            entropy = probability_distribution_new.entropy()
            # estimate new log probabilities using old actions
            actions_log_probability_new = probability_distribution_new.log_prob(actions)
            surrogate_loss = calculate_surrogate_loss(
                    actions_log_probability_old,
                    actions_log_probability_new,
                    epsilon,
                    advantages)
            policy_loss, value_loss = calculate_losses(
                    surrogate_loss,
                    entropy,
                    entropy_coefficient,
                    returns,
                    value_pred)
            optimizer.zero_grad()
            policy_loss.backward()
            value_loss.backward()
            optimizer.step()
            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
    return total_policy_loss / ppo_steps, total_value_loss / ppo_steps


In [6]:
def run_ppo(env, max_episodes, discount_factor, ppo_steps, n_trials, epsilon, ):
    train_rewards = []
    test_rewards = []
    policy_losses = []
    value_losses = []

    actor = Actor(env.n_states, env.n_actions)
    critic = Critic(env.n_states, env.n_actions)
    agent = PPOAgent(env, actor, critic, lr_actor=0.001, lr_critic=0.001, gamma=0.99)
    trajectory_histories = []
    policy_histories = []
    V_histories = []
    
    for episode in range(1, MAX_EPISODES+1):
        train_reward, states, actions, actions_log_probability, advantages, returns = agent.forward_pass()
        policy_loss, value_loss = update_policy(
                agent,
                states,
                actions,
                actions_log_probability,
                advantages,
                returns,
                optimizer,
                PPO_STEPS,
                EPSILON,
                ENTROPY_COEFFICIENT)
        test_reward = evaluate(env_test, agent)
        policy_losses.append(policy_loss)
        value_losses.append(value_loss)
        train_rewards.append(train_reward)
        test_rewards.append(test_reward)
        mean_train_rewards = np.mean(train_rewards[-N_TRIALS:])
        mean_test_rewards = np.mean(test_rewards[-N_TRIALS:])
        mean_abs_policy_loss = np.mean(np.abs(policy_losses[-N_TRIALS:]))
        mean_abs_value_loss = np.mean(np.abs(value_losses[-N_TRIALS:]))
        if episode % PRINT_INTERVAL == 0:
            print(f'Episode: {episode:3} | \
                  Mean Train Rewards: {mean_train_rewards:3.1f} \
                  | Mean Test Rewards: {mean_test_rewards:3.1f} \
                  | Mean Abs Policy Loss: {mean_abs_policy_loss:2.2f} \
                  | Mean Abs Value Loss: {mean_abs_value_loss:2.2f}')
        if mean_test_rewards >= REWARD_THRESHOLD:
            print(f'Reached reward threshold in {episode} episodes')
            break
    plot_train_rewards(train_rewards, REWARD_THRESHOLD)
    plot_test_rewards(test_rewards, REWARD_THRESHOLD)
    plot_losses(policy_losses, value_losses)


In [7]:
# lake_grid = [["G", "H", "F", "F"],
#              ["F", "F", "F", "F"],
#              ["F", "F", "H", "F"],
#              ["F", "H", "S", "F"]]

lake_grid = [["F", "F", "S", "F", "H"],
             ["F", "F", "H", "F", "F"],
             ["F", "F", "F", "G", "F"],
             ["F", "H", "F", "F", "F"],
             ["H", "H", "F", "F", "F"]]

reward_points = {
    "S": 0,
    "G": 1,
    "F": 0,
    "H": 0
}

frozen_lake = FrozenLakeEnvironment(grid=lake_grid,
                                    reward_points=reward_points,
                                    slippery=True)

In [8]:
policy_histories, v_histories, trajectory_histories = train(frozen_lake, n_episodes=5000)

NameError: name 'train' is not defined

In [9]:
policy = policy_histories[-1]
policy

NameError: name 'policy_histories' is not defined

In [10]:
trajectory_histories[0]

NameError: name 'trajectory_histories' is not defined

Render policy

In [11]:
import pandas as pd

In [12]:
len(trajectory_histories)

NameError: name 'trajectory_histories' is not defined

In [13]:
trajectory_histories[3]

NameError: name 'trajectory_histories' is not defined

In [14]:
plot_trajectory_history(frozen_lake, trajectory_histories, policy)

NameError: name 'trajectory_histories' is not defined